# Graphs · Three nodes and two edges
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

**Why this is needed:** coordinates alone cannot tell a program which parts are
connected. A graph stores **nodes** and **edges** separately from their geometry.
A three-node chain is enough to learn the data model used later by trusses and cables.
This lesson stores a load label but does not solve structural equilibrium.

Coordinates use metres; a stored vertical load uses kN.
**Predict:** how many neighbours does the middle node have?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from compas.datastructures import Graph
from compas_notebook import Viewer
from compas_notebook.config import Config

graph = Graph()
graph.add_node("left", x=0, y=0, z=0)
graph.add_node("middle", x=1, y=0, z=0.5)
graph.add_node("right", x=2, y=0, z=0)
graph.add_edge("left", "middle")
graph.add_edge("middle", "right")
graph.node_attribute("middle", "pz", -1.0)  # kN; +z points up
graph.attributes.update({"length_unit": "m", "force_unit": "kN"})
print(graph)
print("Middle neighbours:", graph.neighbors("middle"))
print("Middle vertical load:", graph.node_attribute("middle", "pz"), "kN")


## Node keys are identifiers, not row numbers
COMPAS accepts named node keys. A numerical array needs rows 0, 1, 2.
`node_index()` gives the map between these two worlds. The adjacency matrix has
1 at neighbouring node pairs and 0 elsewhere; its entries carry no length units.


In [ ]:
key_to_row = graph.node_index()
A = np.asarray(graph.adjacency_matrix(rtype="array"))
print("Key → row:", key_to_row)
print("Adjacency:\n", A)
assert graph.number_of_nodes() == 3 and graph.number_of_edges() == 2
assert graph.degree("middle") == 2
assert np.array_equal(A, A.T)
assert A[key_to_row["left"], key_to_row["middle"]] == 1
assert A[key_to_row["left"], key_to_row["right"]] == 0
assert np.allclose(A.sum(axis=1), [graph.degree(key) for key in graph.nodes()])

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [3, -5, 2]
config.view.camera.target = [1, 0, 0.25]
view = Viewer(config=config)
view.scene.add(graph, color="#5552bb", nodesize=0.12)
view.show()


## Predict → change → check
Starting result: **3 nodes, 2 edges**; middle degree **2**.

1. Change the middle node's height and predict whether adjacency changes.
2. On a copy, connect left directly to right. Predict the degree of every node.
3. Explain the difference between the graph's `pz` attribute and an equilibrium result.

The next cell adds a connection on a copy and leaves the reference chain intact.


In [ ]:
trial = graph.copy()
trial.add_edge("left", "right")
assert trial.number_of_edges() == graph.number_of_edges() + 1
assert all(trial.degree(key) == 2 for key in trial.nodes())
print("The copy is a three-edge cycle:", list(trial.edges()))


## Sources
Original minimal example informed by [ITA21 graph data structures](https://github.com/compas-teaching/ITA21/tree/7213b322067efe6102e998e83194e8dbe39a82bf)
and [Seoul 2025 COMPAS](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a/05_compas).
API: [COMPAS Graph](https://compas.dev/compas/2.15.1/api/compas.datastructures.html); native viewer pattern:
[COMPAS JupyterLite graph example](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
